In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
import json
import os
import uuid

from IPython.display import HTML
from google.colab import userdata
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Interrupt
from langsmith import Client as LangSmithClient
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_WORKSPACE_ID"] = userdata.get('LANGSMITH_WORKSPACE_ID')
os.environ["LANGSMITH_PROJECT"] = "LangChain Memory & Human-in-the-Loop"

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

def print_interrupts(interrupts: List[Interrupt]):
    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [ ]:
@tool
def search_travel_options(destination: str) -> str:
    """Call this tool to search for flight and accommodation options for a trip."""
    options = {
        "destination": destination,
        "flights": [
            {"flight_code": "LH170", "departure": "08:10", "price_eur": 189},
            {"flight_code": "FR402", "departure": "09:05", "price_eur": 129},
        ],
        "hotels": [
            {"hotel_name": "Alexander Hub Hotel", "price_eur": 176},
            {"hotel_name": "Spring View Stay", "price_eur": 144},
        ]
    }
    return json.dumps(options, indent=2)

@tool
def book_flight(traveler_name: str, flight_code: str) -> str:
    """Call this tool to book a flight."""
    return f"Booked flight {flight_code} for {traveler_name}"

@tool
def book_hotel(traveler_name: str, hotel_name: str, nights: int) -> str:
    """Call this tool to book a hotel."""
    return f"Booked {nights} night(s) at {hotel_name} for {traveler_name}"

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low"),
    tools=[search_travel_options, book_flight, book_hotel],
    system_prompt="You are a travel operations assistant.",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                book_flight.name: True,
                book_hotel.name: True
            }
        )
    ],
)

In [ ]:
configurable = {
    "thread_id": "berlin_trip_1"
}

In [ ]:
first_run_id = uuid.uuid4();
berlin_trip = agent.invoke(
    input={
        "messages": [HumanMessage("Plan a one-night business trip to Berlin for Dana. Search options, book a morning flight, and reserve one hotel night for up to 150 EUR. If there are many possible options, take the initiative and select the most affordable one.")]
    },
    config={
        "configurable": configurable,
        "run_id": first_run_id
    }
)

In [ ]:
print_conversation(berlin_trip["messages"])
print_interrupts(berlin_trip.get("__interrupt__", []))

In [ ]:
second_run_id = uuid.uuid4()

# NOTE: We expect two blocked tool calls - one too book a flight, and another to book a hotel. Let's approve both of them.
berlin_trip = agent.invoke(
    input=Command(
        resume={
            "decisions": [{ "type": "approve" }, { "type": "approve" }]
        }
    ),
    config={
        "configurable": configurable,
        "run_id": second_run_id
    }
)

In [ ]:
print_conversation(berlin_trip["messages"])

In [ ]:
lang_smith_client = LangSmithClient()
_ = lang_smith_client.create_feedback(run_id=first_run_id, key="user_rating", score=1)
_ = lang_smith_client.create_feedback(run_id=second_run_id, key="user_rating", score=1)